### 🧹 AWS Cleanup Script: Scanned & Deleted Resources

The cleanup script targets **9 core AWS services** across both global and region-specific infrastructure, prioritizing components that incur ongoing hourly or storage charges.

| Service | Specific Resource Scanned | What Gets Deleted |
| :--- | :--- | :--- |
| **S3** *(Global)* | Buckets | Empties all objects, version histories, and delete markers, then deletes the bucket itself. |
| **SageMaker** | Real-time Endpoints | Active inference endpoints incurring hourly compute charges. |
| **SageMaker** | Endpoint Configurations | Saved deployment configurations for model hosting. |
| **SageMaker** | Models | Created SageMaker model entities and container definitions. |
| **EKS** | Managed Kubernetes Clusters | The control plane for EKS clusters. |
| **CloudFormation** | Active Stacks | Automatically updates termination protection to `False` and deletes stack resources (with a `RetainResources` fallback if trapped in `DELETE_FAILED`). |
| **ECR** | Container Repositories | Repositories holding Docker images (forces deletion even if images exist). |
| **Elastic Load Balancing** | Application / Network Load Balancers | Active load balancers (including those auto-provisioned by EKS services). |
| **EC2 / Storage** | Unattached EBS Volumes | Idle block storage volumes sitting in the `available` state. |
| **EC2 / Networking** | Unattached Elastic IPs (EIPs) | Public IPv4 addresses not associated with a running EC2 instance or Load Balancer. |

In [1]:
# =========================================================
# GOOGLE COLAB BASH CELL: AWS CLI ALTERNATIVE
# =========================================================
# Paste this in a separate cell in Google Colab to run the same nuke via AWS CLI

!pip install awscli -q

# Set environment variables using Google Colab secrets
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID") or userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY") or userdata.get("AWS_SECRET_ACCESS_KEY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 19.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [2]:
%%bash
echo "🚀 Starting AWS CLI Cleanup Sweep..."

# 1. Delete all S3 Buckets & Contents
echo "------------------------------------------"
echo "Cleaning S3 Buckets..."
echo "------------------------------------------"
for bucket in $(aws s3api list-buckets --query "Buckets[].Name" --output text); do
  echo "❌ Force deleting bucket: $bucket"
  aws s3 rb "s3://$bucket" --force
done

# 2. Iterate through all enabled EC2 regions
REGIONS=$(aws ec2 describe-regions --query "Regions[].RegionName" --output text --region us-east-1)

for region in $REGIONS; do
  echo "------------------------------------------"
  echo "Scanning Region: $region"
  echo "------------------------------------------"

  # Delete SageMaker MLflow Tracking Servers
  SERVERS=$(aws sagemaker list-mlflow-tracking-servers --region "$region" --query "TrackingServerSummaries[].TrackingServerName" --output text 2>/dev/null)
  for server in $SERVERS; do
    if [ "$server" != "None" ] && [ -n "$server" ]; then
      echo "  ❌ Deleting MLflow Server: $server"
      aws sagemaker delete-mlflow-tracking-server --tracking-server-name "$server" --region "$region"
    fi
  done

  # Delete SageMaker Endpoints
  ENDPOINTS=$(aws sagemaker list-endpoints --region "$region" --query "Endpoints[].EndpointName" --output text 2>/dev/null)
  for ep in $ENDPOINTS; do
    if [ "$ep" != "None" ] && [ -n "$ep" ]; then
      echo "  ❌ Deleting Endpoint: $ep"
      aws sagemaker delete-endpoint --endpoint-name "$ep" --region "$region"
    fi
  done

  # Delete SageMaker Endpoint Configs
  CONFIGS=$(aws sagemaker list-endpoint-configs --region "$region" --query "EndpointConfigs[].EndpointConfigName" --output text 2>/dev/null)
  for cfg in $CONFIGS; do
    if [ "$cfg" != "None" ] && [ -n "$cfg" ]; then
      echo "  ❌ Deleting Endpoint Config: $cfg"
      aws sagemaker delete-endpoint-config --endpoint-config-name "$cfg" --region "$region"
    fi
  done

  # Delete SageMaker Models
  MODELS=$(aws sagemaker list-models --region "$region" --query "Models[].ModelName" --output text 2>/dev/null)
  for model in $MODELS; do
    if [ "$model" != "None" ] && [ -n "$model" ]; then
      echo "  ❌ Deleting Model: $model"
      aws sagemaker delete-model --model-name "$model" --region "$region"
    fi
  done

done

echo "=========================================="
echo "AWS CLI SWEEP COMPLETE!"
echo "=========================================="

🚀 Starting AWS CLI Cleanup Sweep...
------------------------------------------
Cleaning S3 Buckets...
------------------------------------------
❌ Force deleting bucket: lesson2-smex-pipeline-455865672536-us-east-2
delete: s3://lesson2-smex-pipeline-455865672536-us-east-2/lesson2/smex-pipeline/runs/run_20260919-172051.json
delete: s3://lesson2-smex-pipeline-455865672536-us-east-2/lesson2/smex-pipeline/data/processed/train.csv
delete: s3://lesson2-smex-pipeline-455865672536-us-east-2/lesson2/smex-pipeline/data/processed/test.csv
delete: s3://lesson2-smex-pipeline-455865672536-us-east-2/lesson2/smex-pipeline/model/model_20260919-172051.joblib
delete: s3://lesson2-smex-pipeline-455865672536-us-east-2/lesson2/smex-pipeline/data/raw/loan_data.csv
❌ Force deleting bucket: lesson3-advanced-pipeline-455865672536-us-east-2
delete: s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/dev/monitoring/drift_report_20260919-172746.json
delete: s3://lesson3-advanced-pipeline-455865672536-us-

remove_bucket failed: s3://lesson2-smex-pipeline-455865672536-us-east-2 An error occurred (BucketNotEmpty) when calling the DeleteBucket operation: The bucket you tried to delete is not empty. You must delete all versions in the bucket.
remove_bucket failed: s3://lesson4-containerization-455865672536-us-east-2 An error occurred (BucketNotEmpty) when calling the DeleteBucket operation: The bucket you tried to delete is not empty. You must delete all versions in the bucket.


In [3]:
!pip install boto3
import os
import time
import boto3
from botocore.exceptions import BotoCoreError, ClientError

# Set DRY_RUN = False to perform actual deletion!
DRY_RUN = False

def get_colab_secret(key_name: str, required: bool = True):
    """Safely retrieves a secret from Google Colab secrets if available."""
    try:
        from google.colab import userdata
        return os.environ.get(key_name) or userdata.get(key_name)
    except Exception:
        if required:
            print(f"⚠️ Warning: Could not retrieve {key_name} from Colab secrets.")
        return None

def get_all_enabled_regions(session=None) -> list:
    """Dynamically retrieves all enabled EC2 regions for the AWS account."""
    try:
        ec2_client = (session or boto3).client("ec2", region_name="us-east-1")
        response = ec2_client.describe_regions(AllRegions=False)
        return [region["RegionName"] for region in response["Regions"]]
    except Exception as exc:
        print(f"⚠️ Could not fetch regions dynamically ({exc}). Falling back to defaults.")
        return ["eu-north-1", "us-east-1", "us-west-2", "eu-west-1"]

def purge_mlflow_servers(sm_client, region_name: str, dry_run: bool = True) -> int:
    """
    Handles MLflow tracking server cleanup across transitional states (Creating, Deleting, etc.).
    Polls actively until all MLflow servers in the region are completely terminated.
    """
    found_count = 0
    try:
        paginator = sm_client.get_paginator("list_mlflow_tracking_servers")
        servers_to_process = []
        for page in paginator.paginate():
            for server in page.get("TrackingServerSummaries", []):
                servers_to_process.append(server)

        if not servers_to_process:
            return 0

        found_count = len(servers_to_process)

        for server in servers_to_process:
            name = server["TrackingServerName"]
            status = server.get("TrackingServerStatus", "UNKNOWN")
            print(f"  ❌ [{region_name}] SageMaker MLflow Server: {name} (Current Status: {status})")

            if dry_run:
                print(f"    🛡️ [DRY-RUN] Would wait for active status and delete MLflow Server: {name}")
                continue

            # Active deletion and state management loop
            max_attempts = 30  # Wait up to 15 minutes (30 * 30s) per server transition
            attempt = 0

            while attempt < max_attempts:
                try:
                    # Refresh server status
                    desc = sm_client.describe_mlflow_tracking_server(TrackingServerName=name)
                    current_status = desc.get("TrackingServerStatus")
                except ClientError as e:
                    # ResourceNotFoundException indicates successful deletion
                    if e.response["Error"]["Code"] in ["ResourceNotFound", "ValidationException"]:
                        print(f"    ✅ MLflow Server '{name}' successfully terminated.")
                        break
                    raise e

                print(f"    ⏳ [{name}] Status: {current_status}. (Attempt {attempt+1}/{max_attempts})")

                if current_status in ["CREATED", "CREATE_FAILED", "UPDATE_FAILED"]:
                    # Disable model registration prior to deletion if active
                    try:
                        sm_client.update_mlflow_tracking_server(
                            TrackingServerName=name,
                            AutomaticModelRegistration=False
                        )
                    except Exception:
                        pass

                    print(f"    🗑️ Requesting deletion for '{name}'...")
                    try:
                        sm_client.delete_mlflow_tracking_server(TrackingServerName=name)
                    except ClientError as exc:
                        print(f"    ⚠️ Delete call failed ({exc.response['Error']['Message']}). Retrying...")

                elif current_status in ["CREATING", "UPDATING"]:
                    print(f"    ⏳ Server is currently {current_status}. Waiting 30s for state transition...")

                elif current_status in ["DELETING"]:
                    print(f"    ⏳ Deletion already in progress by AWS background tasks. Waiting 30s...")

                elif current_status == "DELETE_FAILED":
                    print(f"    ⚠️ Retrying deletion on failed server '{name}'...")
                    try:
                        sm_client.delete_mlflow_tracking_server(TrackingServerName=name)
                    except Exception as exc:
                        print(f"    ❌ Failed to force re-delete: {exc}")

                time.sleep(30)
                attempt += 1

    except Exception as exc:
        print(f"  Error checking MLflow Servers in {region_name}: {exc}")

    return found_count

def nuke_all_aws_resources(dry_run: bool = True):
    if dry_run:
        print("🛡️  RUNNING IN DRY-RUN MODE: No resources will be deleted.\n")
    else:
        print("💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!\n")

    access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
    secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
    session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

    init_session = boto3.session.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name="us-east-1",
    )

    regions = get_all_enabled_regions(session=init_session)
    total_found = 0

    # Clean up S3 buckets first
    try:
        s3_client = init_session.client("s3")
        s3_resource = init_session.resource("s3")
        buckets = s3_client.list_buckets().get("Buckets", [])
        for b in buckets:
            name = b["Name"]
            print(f"  ❌ Found S3 Bucket: {name}")
            total_found += 1
            if not dry_run:
                bucket = s3_resource.Bucket(name)
                bucket.object_versions.delete()
                bucket.delete()
                print(f"    ✅ Deleted S3 Bucket: {name}")
    except Exception as exc:
        print(f"  Error checking S3 Buckets: {exc}")

    # Region sweep
    for region_name in regions:
        print(f"\n==========================================")
        print(f"Scanning Region: {region_name}")
        print(f"==========================================")
        try:
            session = boto3.session.Session(
                region_name=region_name,
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                aws_session_token=session_token,
            )
            sm = session.client("sagemaker", region_name=region_name)

            # MLflow Server Purge with Polling
            total_found += purge_mlflow_servers(sm, region_name, dry_run=dry_run)

            # Endpoints
            for page in sm.get_paginator("list_endpoints").paginate():
                for ep in page.get("Endpoints", []):
                    total_found += 1
                    if not dry_run:
                        sm.delete_endpoint(EndpointName=ep["EndpointName"])

            # Endpoint Configs
            for page in sm.get_paginator("list_endpoint_configs").paginate():
                for cfg in page.get("EndpointConfigs", []):
                    total_found += 1
                    if not dry_run:
                        sm.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])

            # Models
            for page in sm.get_paginator("list_models").paginate():
                for model in page.get("Models", []):
                    total_found += 1
                    if not dry_run:
                        sm.delete_model(ModelName=model["ModelName"])

        except Exception as exc:
            print(f"[{region_name}] Access failed: {exc}")

    print(f"\n==========================================")
    print(f"SWEEP COMPLETE: Processed {total_found} total resource(s).")
    print(f"==========================================")

# Execute sweep
nuke_all_aws_resources(dry_run=DRY_RUN)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.7 MB/s eta 0:00:00
💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!

  ❌ Found S3 Bucket: lesson2-smex-pipeline-455865672536-us-east-2
    ✅ Deleted S3 Bucket: lesson2-smex-pipeline-455865672536-us-east-2
  ❌ Found S3 Bucket: lesson4-containerization-455865672536-us-east-2
    ✅ Deleted S3 Bucket: lesson4-containerization-455865672536-us-east-2

Scanning Region: ap-south-1

Scanning Region: eu-north-1

Scanning Region: eu-west-3

Scanning Region: eu-west-2

Scanning Region: eu-west-1

Scanning Region: ap-northeast-3

Scanning Region: ap-northeast-2

Scanning Region: ap-northeast-1

Scanning Region: ca-central-1

Scanning Region: sa-east-1

Scanning Region: ap-southeast-1

Scanning Region: ap-southeast-2

Scanning Region: eu-central-1

Scanning Regio

In [4]:
import os
import boto3

def get_bucket_region(s3_client, bucket_name: str) -> str:
    """Detect the exact AWS region where an S3 bucket resides."""
    try:
        location = s3_client.get_bucket_location(Bucket=bucket_name).get("LocationConstraint")
        # AWS returns None for us-east-1 and "EU" for legacy eu-west-1
        if location is None:
            return "us-east-1"
        if location == "EU":
            return "eu-west-1"
        return location
    except Exception:
        return "us-east-1"

def delete_usecase_etl_buckets(dry_run: bool = True):
    try:
        access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
        secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
        session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

        session = boto3.session.Session(
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            aws_session_token=session_token
        )
    except Exception as exc:
        print(f"❌ Could not build AWS session: {exc}")
        return

    # Global client used to scan account buckets across all regions
    global_s3_client = session.client("s3")

    try:
        account_id = session.client("sts").get_caller_identity()["Account"]
    except Exception as exc:
        print(f"❌ Could not resolve AWS account id: {exc}")
        return

    expected = {f"usecase-etl-{i}-{account_id}" for i in (1, 2)}
    buckets = global_s3_client.list_buckets().get("Buckets", [])
    targets = [b["Name"] for b in buckets if b["Name"] in expected or b["Name"].startswith("usecase-etl-")]

    print(f"🗂️ Found {len(targets)} UseCase ETL bucket(s) to remove across all regions.")

    for name in targets:
        # Dynamically resolve region for each bucket before deletion operations
        bucket_region = get_bucket_region(global_s3_client, name)
        print(f"📍 Target '{name}' resides in region: {bucket_region}")

        # Instantiate region-specific resource to execute deletion cleanly
        regional_s3_resource = session.resource("s3", region_name=bucket_region)
        empty_and_delete_s3_bucket(regional_s3_resource, name, dry_run=dry_run)

    print(f"✅ UseCase ETL bucket cleanup complete ({len(targets)} bucket(s) handled).")

delete_usecase_etl_buckets(dry_run=DRY_RUN)

🗂️ Found 0 UseCase ETL bucket(s) to remove across all regions.
✅ UseCase ETL bucket cleanup complete (0 bucket(s) handled).


In [5]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-20 00:50:23
